In [1]:
# Imports
import numpy as np
import pandas as pd

In [2]:
rng = np.random.default_rng(42)
N = 10000

In [3]:
# --- Demographics ---
age = np.clip(rng.normal(38, 12, N).round(), 18, 75).astype(int)

gender = rng.choice(["Male", "Female", "Other"], N, p=[0.49, 0.48, 0.03])

location = rng.choice(
    ["Mumbai", "Delhi", "Bengaluru", "Pune", "Hyderabad", "Chennai", "Kolkata", "Ahmedabad"],
    N, 
    p=[0.16, 0.16, 0.16, 0.12, 0.12, 0.10, 0.10, 0.08]
)

In [4]:
print(age)
print(gender)
print(location)

[42 26 47 ... 39 51 36]
['Female' 'Male' 'Female' ... 'Female' 'Female' 'Female']
['Delhi' 'Hyderabad' 'Bengaluru' ... 'Pune' 'Mumbai' 'Pune']


In [5]:
# --- Subscription ---
subscription = rng.choice(["Basic", "Standard", "Premium"], N, p=[0.40, 0.38, 0.22])

contract = rng.choice(["Month-to-Month", "One Year", "Two Year"], N, p=[0.55, 0.28, 0.17])

payment = rng.choice(
    ["Credit Card", "Debit Card", "UPI", "Net Banking", "Wallet"], 
    N,
    p=[0.25, 0.20, 0.35, 0.12, 0.08]
)

tenure = np.clip(rng.gamma(2.2, 14, N).round(), 1, 72).astype(int)

base_price = pd.Series(subscription).map({"Basic": 299, "Standard": 599, "Premium": 999}).values

monthly = np.round(base_price * rng.uniform(0.9, 1.15, N), 2)

total = np.round(monthly * tenure * rng.uniform(0.92, 1.0, N), 2)

In [6]:
print(subscription)
print(contract)
print(payment)
print(tenure)
print(base_price)
print(monthly)
print(total)

['Premium' 'Basic' 'Premium' ... 'Premium' 'Premium' 'Basic']
['One Year' 'Two Year' 'One Year' ... 'Month-to-Month' 'Two Year'
 'One Year']
['Credit Card' 'Debit Card' 'Net Banking' ... 'Debit Card' 'Debit Card'
 'UPI']
[16 31 28 ... 41 59 55]
[999 299 999 ... 999 999 299]
[1148.12  331.28 1107.48 ... 1009.77 1087.95  285.66]
[18313.89 10147.09 28768.6  ... 40109.21 60669.97 15348.1 ]


In [7]:
# --- Behaviour ---
login_freq = np.clip(rng.poisson(14, N) - (contract == "Month-to-Month") * 2, 0, 40)  # logins / month

last_login = np.clip(rng.exponential(9, N).round(), 0, 120).astype(int)

usage = np.clip(rng.gamma(3, 6, N) * (login_freq / 14 + 0.3), 0.5, 150).round(1)

tickets = rng.poisson(1.5, N)

complaints = np.clip(rng.poisson(0.4 + 0.25 * tickets), 0, 8)

discount = rng.choice([0, 5, 10, 15, 20, 25], N, p=[0.45, 0.15, 0.15, 0.1, 0.1, 0.05])

In [8]:
print(login_freq)
print(last_login)
print(usage)
print(tickets)
print(complaints)
print(discount)

[14 17 17 ...  8 17 16]
[22  6  1 ... 37 12  8]
[47.5 26.3 39.7 ...  0.8  4.1 40.2]
[4 2 1 ... 1 1 1]
[2 1 0 ... 0 1 0]
[ 0  0  0 ... 25 15 20]


In [9]:
# --- Satisfaction (1-5), driven by experience ---
sat_raw = 4.2 - 0.35 * complaints - 0.12 * tickets - 0.01 * last_login + rng.normal(0, 0.6, N)

satisfaction = np.clip(sat_raw.round(), 1, 5).astype(int)

print(satisfaction)

[2 4 4 ... 4 4 4]


In [10]:
# --- Churn logic (hidden relationship the model should discover) ---
z = (-1.4
     + 1.0 * (contract == "Month-to-Month")
     - 0.9 * (contract == "Two Year")
     - 0.03 * tenure
     + 0.035 * last_login
     - 0.045 * login_freq
     + 0.35 * complaints
     + 0.12 * tickets
     - 0.55 * (satisfaction - 3)
     + 0.0008 * (monthly - 600)
     - 0.02 * discount
     + 0.25 * (payment == "Wallet")
     + 0.15 * (subscription == "Premium") * (satisfaction <= 2)
     + rng.normal(0, 0.5, N))

prob = 1 / (1 + np.exp(-z))
churn = (rng.uniform(0, 1, N) < prob).astype(int)

In [11]:
print(churn)

[1 0 0 ... 0 0 0]


In [21]:
df = pd.DataFrame({
    "Customer_ID": [f"CUST{100000 + i}" for i in range(N)],
    "Age": age, 
    "Gender": gender, 
    "Location": location,
    "Tenure_Months": tenure, 
    "Monthly_Charges": monthly, 
    "Total_Charges": total,
    "Subscription_Type": subscription, 
    "Payment_Method": payment,
    "Login_Frequency": login_freq, 
    "Support_Tickets": tickets, 
    "Complaints": complaints,
    "Discount_Received": discount, 
    "Contract_Type": contract,
    "Last_Login_Days": last_login, 
    "Usage_Hours": usage,
    "Customer_Satisfaction": satisfaction, 
    "Churn": churn,
})

In [22]:
df.head()

,Customer_ID,Age,Gender,Location,Tenure_Months,Monthly_Charges,Total_Charges,Subscription_Type,Payment_Method,Login_Frequency,Support_Tickets,Complaints,Discount_Received,Contract_Type,Last_Login_Days,Usage_Hours,Customer_Satisfaction,Churn
0,CUST100000,42,Female,Delhi,16,1148.12,18313.89,Premium,Credit Card,14,4,2,0,One Year,22,47.5,2,1
1,CUST100001,26,Male,Hyderabad,31,331.28,10147.09,Basic,Debit Card,17,2,1,0,Two Year,6,26.3,4,0
2,CUST100002,47,Female,Bengaluru,28,1107.48,28768.60,Premium,Net Banking,17,1,0,0,One Year,1,39.7,4,0
3,CUST100003,49,Male,Ahmedabad,26,953.48,24395.43,Premium,UPI,21,0,0,10,Two Year,21,29.8,5,0
4,CUST100004,18,Male,Kolkata,29,601.22,16855.35,Standard,Debit Card,10,1,0,0,One Year,6,15.3,4,0


In [23]:
def pick(frac):
    """Random row positions covering `frac` of the dataset (no repeats)."""
    return rng.choice(N, int(N * frac), replace=False)

In [24]:
missing_plan = {
    "Gender": 0.03, 
    "Location": 0.03, 
    "Payment_Method": 0.02,
    "Contract_Type": 0.04, 
    "Subscription_Type": 0.03,
    "Tenure_Months": 0.03, 
    "Monthly_Charges": 0.02,
    "Login_Frequency": 0.03, 
    "Last_Login_Days": 0.02,
    "Support_Tickets": 0.03, 
    "Discount_Received": 0.02,
}

In [ ]:
# --- adding missing values in a few columns ---
for col, frac in missing_plan.items():
    if pd.api.types.is_numeric_dtype(df[col]):
        df[col] = df[col].astype(float)
    df.loc[pick(frac), col] = np.nan

In [ ]:
# --- placeholders that secretly mean "missing" ---
for col, token in [("Gender", "Unknown"), ("Location", "?"), ("Payment_Method", "N/A")]:
    df.loc[pick(0.01), col] = token
df.loc[pick(0.01), "Last_Login_Days"] = -1

In [26]:
# --- Realistic messiness: 10% missing values in a few columns ---
for col in ["Age", "Total_Charges", "Customer_Satisfaction", "Usage_Hours"]:
    idx = rng.choice(N, int(N * 0.1), replace=False)
    df[col] = df[col].astype(float)
    df.loc[idx, col] = np.nan

In [27]:
# --- Outliers ---
def inject(col, values, n):
    idx = rng.choice(N, n, replace=False)
    df.loc[idx, col] = rng.choice(values, n)

inject("Age", [-5, 0, 120, 150, 999], 15)
inject("Tenure_Months", [0, -3, 240, 500], 12)
inject("Monthly_Charges", [-599, 0, 9999, 59900], 12)
inject("Usage_Hours", [-10, 0, 500, 2400], 12)
inject("Customer_Satisfaction", [0, 6, 7, 10], 20)      # valid range is 1-5
inject("Support_Tickets", [-2, 40, 99], 8)
inject("Discount_Received", [-10, 60, 100, 150], 10)

In [28]:
# Total_Charges far below Monthly_Charges x Tenure --
idx = pick(0.01)
df.loc[idx, "Total_Charges"] = (df.loc[idx, "Monthly_Charges"] * 0.1).round(2)

In [29]:
# ------ case, spaces, alternate spellings ------
def scramble(col, frac, alt=None):
    for i in pick(frac):
        v = df.at[i, col]
        if not isinstance(v, str):
            continue
        choice = rng.integers(0, 4)
        if choice == 0:
            df.at[i, col] = v.lower()
        elif choice == 1:
            df.at[i, col] = v.upper()
        elif choice == 2:
            df.at[i, col] = f" {v} "
        elif alt and v in alt:
            df.at[i, col] = alt[v]
 
scramble("Gender", 0.08, {"Male": "M", "Female": "F"})
scramble("Location", 0.08, {"Bengaluru": "Bangalore", "Mumbai": "Bombay", "Kolkata": "Calcutta"})
scramble(
    "Payment_Method", 0.08, {"UPI": "Upi", "Credit Card": "CreditCard", "Debit Card": "Debit", "Net Banking": "NetBanking"}
)
scramble("Contract_Type", 0.08, {"Month-to-Month": "Month to Month", "One Year": "1 Year", "Two Year": "2 Year"})
scramble("Subscription_Type", 0.06)

In [30]:
# -------- numbers stored as text --------
df["Total_Charges"] = df["Total_Charges"].astype(object)

df.loc[pick(0.03), "Total_Charges"] = " "                # blank space, not NaN

for i in pick(0.02):
    v = df.at[i, "Total_Charges"]
    if not isinstance(v, str) and pd.notna(v):
        df.at[i, "Total_Charges"] = f"₹{v:,.2f}"      # e.g. "₹12,345.60"
 
df["Age"] = df["Age"].astype(object)
words = ["thirty-five", "forty", "twenty five", "N/A"]

for i in pick(0.004):
    df.at[i, "Age"] = rng.choice(words)

In [31]:
# ------- exact copies + same Customer_ID with conflicting values -------
exact = df.sample(60, random_state=7)
conflict = df.sample(10, random_state=11).copy()

conflict["Support_Tickets"] = pd.to_numeric(conflict["Support_Tickets"], errors="coerce").fillna(0) + 1
df = pd.concat([df, exact, conflict], ignore_index=True)
 
# ------- shuffle rows so duplicates are not sitting next to each other -------
df = df.sample(frac=1, random_state=7).reset_index(drop=True)

In [ ]:
df.to_csv("data/raw/customer_churn_dataset.csv", index=False)
print(df.shape)

(10070, 18)


In [33]:
print("Churn rate:", round(df["Churn"].mean() * 100, 1), "%")

Churn rate: 15.9 %


In [34]:
print(df.isna().sum())

Customer_ID                 0
Age                      1001
Gender                      0
Location                    0
Tenure_Months               0
Monthly_Charges             0
Total_Charges             967
Subscription_Type           0
Payment_Method              0
Login_Frequency             0
Support_Tickets             0
Complaints                  0
Discount_Received           0
Contract_Type               0
Last_Login_Days             0
Usage_Hours              1008
Customer_Satisfaction    1006
Churn                       0
dtype: int64


In [36]:
print(df.duplicated().sum())

60


In [38]:
print(df['Customer_ID'].duplicated(keep=False).sum())

140


In [40]:
df.head()

,Customer_ID,Age,Gender,Location,Tenure_Months,Monthly_Charges,Total_Charges,Subscription_Type,Payment_Method,Login_Frequency,Support_Tickets,Complaints,Discount_Received,Contract_Type,Last_Login_Days,Usage_Hours,Customer_Satisfaction,Churn
0,CUST104939,27.0,Female,Ahmedabad,12,1106.24,12694.02,Premium,Debit Card,16,1,0,0,Month-to-Month,10,46.7,4.0,0
1,CUST108673,52.0,Male,Hyderabad,17,621.18,62.12,Standard,UPI,13,1,1,20,Month-to-Month,12,45.4,5.0,0
2,CUST103277,18.0,Male,?,31,297.43,9149.23,Basic,UPI,14,3,1,5,Month-to-Month,11,19.4,4.0,1
3,CUST105552,45.0,Other,Kolkata,26,594.26,14894.33,Standard,UPI,6,4,3,0,Month-to-Month,8,NaN,1.0,0
4,CUST108857,49.0,Female,Mumbai,28,570.39,15385.08,Standard,Debit Card,10,0,0,5,One Year,23,26.4,4.0,0
